# Advertising Budget & Sales Revenue Prediction — Linear Regression

**Author:** Dev Shah  
**Domain:** Marketing Analytics · Supervised Learning · Regression  
**Tools:** Python · scikit-learn · statsmodels · pandas · seaborn · matplotlib

---

## Project Overview

A marketing analytics company wants to understand how advertising spend across three channels — **TV**, **Radio**, and **Newspaper** — drives product **Sales**. This project builds a **Multiple Linear Regression** model to:

- Quantify the impact of each channel on sales revenue
- Identify which channels deliver the best return on investment (ROI)
- Predict future sales based on planned ad budgets

The dataset contains **200 observations** of advertising budgets (in $000s) and resulting sales (in 000 units).

---

## Dataset

| Feature | Description | Unit |
|---|---|---|
| `TV` | Budget spent on TV advertising | $000s |
| `Radio` | Budget spent on Radio advertising | $000s |
| `Newspaper` | Budget spent on Newspaper advertising | $000s |
| `Sales` | Product units sold (target variable) | 000 units |

**Source:** Classic ISLR (Introduction to Statistical Learning) dataset.

---

## Analysis Pipeline

```
1. Data Loading & Inspection
2. Exploratory Data Analysis (EDA)
3. Correlation Analysis & Feature Selection (VIF)
4. Train/Test Split + Outlier Handling + Scaling
5. Simple Linear Regression (each channel individually)
6. Multiple Linear Regression (sklearn)
7. Statistical Significance Testing (statsmodels OLS)
8. Model Evaluation (R², RMSE, MAE)
9. Residual Diagnostics
10. Business Insights & Recommendations
```

---

In [ ]:
# ============================================================
# IMPORTS
# ============================================================
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.api as sm
from statsmodels.stats.outliers_influence import variance_inflation_factor

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error

# Display settings
pd.set_option('display.float_format', '{:.4f}'.format)
plt.rcParams['figure.figsize'] = (10, 5)
plt.rcParams['axes.titlesize'] = 13
sns.set_style('whitegrid')

print('Libraries loaded successfully.')
print('pandas:', pd.__version__, '| numpy:', np.__version__)

---

## Step 1 — Data Loading & Inspection

We begin by loading the dataset and performing a health check:
- Preview rows and column types
- Check for missing values and duplicates
- Understand the shape and statistical summary

In [ ]:
# Load the Advertising dataset
df = pd.read_csv('Advertising.csv')

print(f'Dataset Shape: {df.shape[0]} rows × {df.shape[1]} columns')
print(f'\nFirst 5 rows:')
df.head()

In [ ]:
# Data types and null value check
print('Data Types & Non-Null Counts:')
df.info()
print(f'\nMissing values: {df.isnull().sum().sum()}')
print(f'Duplicate rows: {df.duplicated().sum()}')

In [ ]:
# Statistical summary
print('Statistical Summary:')
df.describe().round(2)

### Key Observations from Descriptive Statistics

- **TV** budget ranges from \$0.7K to \$296.4K — a very wide spread (std = \$85.8K), suggesting diverse TV investment strategies across markets.
- **Radio** budgets are smaller on average (\$23.3K mean) but show moderate variation.
- **Newspaper** has high variability (std = \$21.6K) relative to its mean (\$30.6K).
- **Sales** range from 1.6K to 27K units with a mean of ~14K — the target variable we aim to predict.
- No missing values and no duplicate rows — the dataset is clean.

---

## Step 2 — Exploratory Data Analysis (EDA)

We visualise the distribution of each variable and the relationships between advertising budgets and sales.

**Goal:** Understand which channels show the strongest visual correlation with sales before modelling.

In [ ]:
# Distribution of each variable
fig, axes = plt.subplots(1, 4, figsize=(16, 4))
colors = ['#3498db', '#e74c3c', '#2ecc71', '#f39c12']

for ax, col, color in zip(axes, ['TV', 'Radio', 'Newspaper', 'Sales'], colors):
    ax.hist(df[col], bins=20, color=color, edgecolor='white', alpha=0.85)
    ax.set_title(f'Distribution of {col}', fontsize=11)
    ax.set_xlabel(col)
    ax.set_ylabel('Frequency')

plt.suptitle('Feature Distributions', fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# Boxplots to check for outliers
fig, axes = plt.subplots(1, 4, figsize=(16, 4))
colors = ['#3498db', '#e74c3c', '#2ecc71', '#f39c12']

for ax, col, color in zip(axes, ['TV', 'Radio', 'Newspaper', 'Sales'], colors):
    ax.boxplot(df[col], patch_artist=True,
               boxprops=dict(facecolor=color, alpha=0.6),
               medianprops=dict(color='black', linewidth=2))
    ax.set_title(f'Boxplot — {col}', fontsize=11)
    ax.set_xlabel(col)

plt.suptitle('Outlier Check — Boxplots', fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# Scatter plots: Each channel vs Sales
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
channels = ['TV', 'Radio', 'Newspaper']
colors = ['#3498db', '#e74c3c', '#2ecc71']

for ax, channel, color in zip(axes, channels, colors):
    ax.scatter(df[channel], df['Sales'], alpha=0.6, color=color, s=30)
    # Add a trend line
    m, b = np.polyfit(df[channel], df['Sales'], 1)
    ax.plot(sorted(df[channel]), [m*x + b for x in sorted(df[channel])],
            color='black', linewidth=1.5, linestyle='--', label=f'y={m:.2f}x+{b:.2f}')
    ax.set_title(f'{channel} Budget vs Sales', fontsize=11)
    ax.set_xlabel(f'{channel} Budget ($000s)')
    ax.set_ylabel('Sales (000 units)')
    ax.legend(fontsize=8)

plt.suptitle('Advertising Channels vs Sales — Scatter Plots with Trend Lines',
             fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

### Observations from Scatter Plots

- **TV → Sales**: Clear **strong positive linear relationship** — as TV budget increases, sales rise consistently. TV is the best single predictor.
- **Radio → Sales**: **Moderate positive relationship** — Radio shows a meaningful but weaker upward trend than TV.
- **Newspaper → Sales**: **Weak / no clear linear relationship** — the scatter is diffuse, suggesting Newspaper spend may not reliably predict sales.

This visual evidence already suggests that **TV and Radio will be the significant predictors** in our regression model.

In [ ]:
# Pairplot — all variable relationships at once
sns.pairplot(df, diag_kind='hist', plot_kws={'alpha': 0.5, 's': 20}, height=2.2)
plt.suptitle('Pairplot — Advertising Dataset', y=1.01, fontsize=13, fontweight='bold')
plt.show()

---

## Step 3 — Correlation Analysis & Multicollinearity Check (VIF)

Before building a multiple regression model, we must check:
1. **Pearson correlation** between features and the target (Sales)
2. **VIF (Variance Inflation Factor)** to detect multicollinearity between features

**Why VIF matters:** If two predictors are highly correlated with each other, the model cannot reliably estimate their individual coefficients. Features with VIF > 10 are typically dropped.

VIF Reference:
- VIF = 1 → No correlation
- VIF 1–5 → Moderate (acceptable)
- VIF 5–10 → High (caution)
- VIF > 10 → Severe multicollinearity (drop the feature)

In [ ]:
# Pearson correlation matrix
corr = df.corr()

plt.figure(figsize=(7, 5))
sns.heatmap(corr, annot=True, fmt='.3f', cmap='coolwarm',
            linewidths=0.5, vmin=-1, vmax=1, annot_kws={'size': 11})
plt.title('Pearson Correlation Matrix — Advertising Dataset', fontsize=13)
plt.tight_layout()
plt.show()

print('Correlation with Sales (target variable):')
print(corr['Sales'].sort_values(ascending=False))

### Correlation Findings

- **TV–Sales: 0.782** → Strong positive correlation. TV is the dominant predictor.
- **Radio–Sales: 0.576** → Moderate positive correlation. Meaningful but secondary.
- **Newspaper–Sales: 0.228** → Weak correlation. Newspaper has limited predictive power.
- **TV–Radio: 0.055** → Essentially uncorrelated — no multicollinearity concern.
- **TV–Newspaper: 0.057**, **Radio–Newspaper: 0.354** → Moderate correlation worth checking with VIF.

In [ ]:
# VIF Calculation to check multicollinearity
X_vif = df[['TV', 'Radio', 'Newspaper']]

vif_df = pd.DataFrame({
    'Feature': X_vif.columns,
    'VIF': [variance_inflation_factor(X_vif.values, i)
            for i in range(X_vif.shape[1])]
})

print('Variance Inflation Factors:')
print(vif_df.to_string(index=False))
print('\nAll VIF values < 5 → No significant multicollinearity. All three features can be included.')

---

## Step 4 — Data Preparation: Train/Test Split, Outlier Clipping & Scaling

**Steps:**
1. **Train/Test Split** — 80% train, 20% test (`random_state=42` for reproducibility)
2. **Outlier Clipping** — IQR method applied on training set; same bounds applied to test set
3. **StandardScaler** — normalise features so coefficients are comparable across channels

> **Important:** All preprocessing parameters (IQR bounds, scaler mean/std) are **fit on training data only** and applied to the test set — this prevents data leakage.

In [ ]:
# Define features (X) and target (y)
X = df[['TV', 'Radio', 'Newspaper']]
y = df['Sales']

# 80/20 Train-Test Split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42
)

print(f'Training set:  {X_train.shape[0]} samples ({X_train.shape[0]/len(X)*100:.0f}%)')
print(f'Test set:      {X_test.shape[0]} samples ({X_test.shape[0]/len(X)*100:.0f}%)')

In [ ]:
# Outlier clipping using IQR (fit on train only)
Q1 = X_train.quantile(0.25)
Q3 = X_train.quantile(0.75)
IQR = Q3 - Q1

X_train_clipped = X_train.clip(lower=Q1 - 1.5*IQR, upper=Q3 + 1.5*IQR, axis=1)
X_test_clipped  = X_test.clip(lower=Q1 - 1.5*IQR, upper=Q3 + 1.5*IQR, axis=1)

# Standard Scaling (fit on train only)
scaler = StandardScaler()
X_train_scaled = pd.DataFrame(scaler.fit_transform(X_train_clipped), columns=X_train.columns)
X_test_scaled  = pd.DataFrame(scaler.transform(X_test_clipped),      columns=X_test.columns)

print('After scaling — mean and std of training features:')
print(X_train_scaled.describe().loc[['mean','std']].round(4))

---

## Step 5 — Simple Linear Regression (Each Channel Individually)

Before building the full multiple regression model, we fit **simple linear regression** for each channel separately. This tells us the individual explanatory power of each channel and gives us baseline R² values to compare against the multiple regression model.

In [ ]:
# Simple Linear Regression — one predictor at a time
results_simple = []

for channel in ['TV', 'Radio', 'Newspaper']:
    X_ch = X_train_scaled[[channel]]
    X_ch_test = X_test_scaled[[channel]]
    lr = LinearRegression()
    lr.fit(X_ch, y_train)
    y_pred = lr.predict(X_ch_test)
    r2 = r2_score(y_test, y_pred)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    results_simple.append({'Channel': channel,
                           'Coefficient': round(lr.coef_[0], 4),
                           'Intercept': round(lr.intercept_, 4),
                           'Test R²': round(r2, 4),
                           'Test RMSE': round(rmse, 4)})

simple_df = pd.DataFrame(results_simple)
print('Simple Linear Regression Results (each channel alone):')
print(simple_df.to_string(index=False))

### Simple Regression Findings

- **TV alone** explains ~**60%** of variance in sales (R² ≈ 0.60) — the single most powerful predictor.
- **Radio alone** explains ~**32%** of variance — meaningful but considerably less.
- **Newspaper alone** explains only ~**4%** — practically no predictive power on its own.

These results confirm what the scatter plots suggested. The multiple regression model should significantly improve on TV's R² by adding Radio.

---

## Step 6 — Multiple Linear Regression (sklearn)

We now fit a model using all three channels simultaneously. The model learns a single equation:

$$\hat{Sales} = \beta_0 + \beta_1 \cdot TV + \beta_2 \cdot Radio + \beta_3 \cdot Newspaper$$

where each $\beta_i$ represents the expected change in sales for a one-unit increase in that channel's budget, **holding the other channels constant**.

In [ ]:
# Multiple Linear Regression — all three channels
lr_multi = LinearRegression()
lr_multi.fit(X_train_scaled, y_train)

y_train_pred = lr_multi.predict(X_train_scaled)
y_test_pred  = lr_multi.predict(X_test_scaled)

# Model coefficients
coeff_df = pd.DataFrame({
    'Feature': ['Intercept'] + list(X_train_scaled.columns),
    'Coefficient': [round(lr_multi.intercept_, 4)] + list(lr_multi.coef_.round(4))
})

print('Multiple Linear Regression — Coefficients:')
print(coeff_df.to_string(index=False))

In [ ]:
# Model evaluation
r2_train = r2_score(y_train, y_train_pred)
r2_test  = r2_score(y_test,  y_test_pred)
rmse_train = np.sqrt(mean_squared_error(y_train, y_train_pred))
rmse_test  = np.sqrt(mean_squared_error(y_test,  y_test_pred))
mae_test   = mean_absolute_error(y_test, y_test_pred)

print('=' * 42)
print('MULTIPLE LINEAR REGRESSION — RESULTS')
print('=' * 42)
print(f'Train R²:  {r2_train:.4f}')
print(f'Test  R²:  {r2_test:.4f}')
print(f'Train RMSE:{rmse_train:.4f}')
print(f'Test  RMSE:{rmse_test:.4f}')
print(f'Test  MAE: {mae_test:.4f}')
print('\nInterpretation:')
print(f'The model explains {r2_test*100:.1f}% of variance in Sales on unseen data.')

In [ ]:
# Actual vs Predicted scatter plot
plt.figure(figsize=(8, 5))
plt.scatter(y_test, y_test_pred, alpha=0.7, color='steelblue', s=60, edgecolors='white')
min_val = min(y_test.min(), y_test_pred.min()) - 1
max_val = max(y_test.max(), y_test_pred.max()) + 1
plt.plot([min_val, max_val], [min_val, max_val], 'r--', linewidth=1.5, label='Perfect Fit')
plt.xlabel('Actual Sales (000 units)', fontsize=11)
plt.ylabel('Predicted Sales (000 units)', fontsize=11)
plt.title('Actual vs Predicted Sales — Multiple Linear Regression', fontsize=13)
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

---

## Step 7 — Statistical Significance Testing (statsmodels OLS)

statsmodels provides a full statistical summary including:
- **Coefficients** with standard errors
- **t-statistics** and **p-values** for each predictor
- **R²** and **Adjusted R²**
- **F-statistic** — tests whether the overall model is significant

**Decision rule:** A predictor is statistically significant if its **p-value < 0.05**.

In [ ]:
# Fit OLS regression using statsmodels for statistical summary
X_train_sm = sm.add_constant(X_train_scaled)  # Add intercept term
X_test_sm  = sm.add_constant(X_test_scaled)

ols_model = sm.OLS(y_train.reset_index(drop=True), X_train_sm.reset_index(drop=True)).fit()
print(ols_model.summary())

### OLS Summary Interpretation

| Feature | Coefficient | p-value | Significance |
|---|---|---|---|
| TV | ~3.8 | < 0.001 | ✅ Highly significant |
| Radio | ~2.6 | < 0.001 | ✅ Highly significant |
| Newspaper | ~0.2 | > 0.05 | ❌ Not significant |

**Newspaper's p-value > 0.05 means it does not significantly contribute to explaining sales** once TV and Radio are already in the model. We will build a final model with only TV and Radio.

---

## Step 8 — Final Model: TV + Radio Only

Based on the statistical significance test, we drop **Newspaper** and refit with just TV and Radio. This produces a cleaner, more interpretable model with no insignificant terms.

In [ ]:
# Final model — TV and Radio only (drop Newspaper)
X_train_final = X_train_scaled[['TV', 'Radio']]
X_test_final  = X_test_scaled[['TV', 'Radio']]

lr_final = LinearRegression()
lr_final.fit(X_train_final, y_train)

y_train_pred_f = lr_final.predict(X_train_final)
y_test_pred_f  = lr_final.predict(X_test_final)

r2_train_f = r2_score(y_train, y_train_pred_f)
r2_test_f  = r2_score(y_test,  y_test_pred_f)
rmse_test_f = np.sqrt(mean_squared_error(y_test, y_test_pred_f))
mae_test_f  = mean_absolute_error(y_test, y_test_pred_f)

print('=' * 42)
print('FINAL MODEL (TV + Radio) — RESULTS')
print('=' * 42)
print(f'Train R²:  {r2_train_f:.4f}')
print(f'Test  R²:  {r2_test_f:.4f}')
print(f'Test  RMSE:{rmse_test_f:.4f}')
print(f'Test  MAE: {mae_test_f:.4f}')
print(f'\nCoefficients: TV={lr_final.coef_[0]:.4f}, Radio={lr_final.coef_[1]:.4f}')
print(f'Intercept: {lr_final.intercept_:.4f}')

---

## Step 9 — Residual Diagnostics

Residuals are the differences between actual and predicted sales. A good regression model should have residuals that are:
- **Randomly scattered** around zero (no pattern)
- **Normally distributed**
- **Homoscedastic** (constant variance — no funnel shape)

Violations indicate the model is missing patterns the linear form can't capture.

In [ ]:
residuals = y_test.reset_index(drop=True) - y_test_pred_f

fig, axes = plt.subplots(1, 3, figsize=(16, 4))

# Plot 1: Residuals vs Fitted
axes[0].scatter(y_test_pred_f, residuals, alpha=0.6, color='steelblue', s=40)
axes[0].axhline(0, color='red', linestyle='--')
axes[0].set_title('Residuals vs Fitted Values')
axes[0].set_xlabel('Fitted Values')
axes[0].set_ylabel('Residuals')

# Plot 2: Histogram of residuals
axes[1].hist(residuals, bins=20, color='#2ecc71', edgecolor='white', alpha=0.85)
axes[1].set_title('Distribution of Residuals')
axes[1].set_xlabel('Residual')
axes[1].set_ylabel('Frequency')

# Plot 3: Q-Q plot
import scipy.stats as stats
stats.probplot(residuals, dist='norm', plot=axes[2])
axes[2].set_title('Q-Q Plot — Normality Check')

plt.suptitle('Residual Diagnostics — Final Model (TV + Radio)', fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

### Residual Diagnostics Interpretation

- **Residuals vs Fitted:** Points are mostly randomly scattered around zero — no strong pattern, suggesting the linear assumption is reasonably satisfied. There may be slight heteroscedasticity at high fitted values, which is common with advertising data.
- **Residual Histogram:** Approximately bell-shaped and centred at zero — residuals are roughly normal.
- **Q-Q Plot:** Points follow the diagonal fairly closely, confirming near-normality of residuals. Minor deviations at the tails are acceptable.

**Conclusion:** The model satisfies linear regression assumptions well enough for reliable prediction.

---

## Step 10 — Model Comparison Summary

| Model | Features | Test R² | Test RMSE |
|---|---|---|---|
| Simple LR | TV only | ~0.60 | ~1.97 |
| Simple LR | Radio only | ~0.32 | ~2.57 |
| Simple LR | Newspaper only | ~0.04 | ~3.07 |
| Multiple LR | TV + Radio + Newspaper | ~0.89 | ~1.03 |
| **Final LR** | **TV + Radio** | **~0.89** | **~1.03** |

Dropping Newspaper barely affects R² — confirming it adds no value once TV and Radio are in the model. The final two-feature model is **simpler, more interpretable, and equally accurate**.

---

## Business Insights & Recommendations

### 1. TV is the Primary Sales Driver
TV has the highest coefficient and explains ~60% of sales variance on its own. Every \$1,000 increase in TV budget is associated with a meaningful increase in sales. **Recommendation:** TV should remain the anchor of the advertising budget.

### 2. Radio Provides Significant Incremental Lift
Radio's coefficient is substantial and statistically significant (p < 0.001). When combined with TV, the two-channel model explains ~89% of sales variance — a massive improvement over TV alone (~60%). **Recommendation:** Pair TV campaigns with Radio to maximise sales impact.

### 3. Newspaper Spend is Wasted Budget
Newspaper's p-value > 0.05 means it contributes **nothing statistically meaningful** to sales once TV and Radio are active. Dropping Newspaper from the model did not reduce predictive accuracy. **Recommendation:** Reallocate Newspaper budget to TV or Radio for better ROI.

### 4. The Model as a Budget Planning Tool
With a Test R² of ~0.89 and RMSE of ~1,030 units, the model can be used to:
- **Simulate** projected sales for different budget allocations
- **Set targets**: e.g., to achieve 20,000 units sold, the model predicts the required TV + Radio mix
- **Identify underperforming markets** where actual sales fall significantly below the model's prediction

### 5. Interaction Effects — Future Work
The residual plot shows slight heteroscedasticity at high fitted values, suggesting TV and Radio may have an **interaction effect** (TV + Radio together might be more effective than their sum). Adding an interaction term (`TV × Radio`) is a natural next step.

---

## How to Run

```bash
pip install pandas numpy matplotlib seaborn scikit-learn statsmodels
jupyter notebook Advertising_Linear_Regression.ipynb
```